# Neural Network Benchmark

- A feedforward neural network was implemented as a benchmark deep learning model for the multi-class classification task.
- The dataset contains 500,000 samples, 7 numerical features, and 2 categorical features.
- Numerical features were standardized using StandardScaler within each cross-validation fold.
- Categorical features were processed using embedding layers to learn compact feature representations.
- The network architecture consists of three hidden layers with 256, 128, and 64 neurons.
- Batch Normalization and ReLU activation were applied after each hidden layer.
- Dropout regularization was used to reduce overfitting.
- The model was trained using the Adam optimizer and sparse categorical cross-entropy loss.
- Stratified K-Fold cross-validation was used to ensure consistency with other models.
- Model performance was evaluated using Balanced Accuracy, and out-of-fold predictions were used to calculate the final score.

In [ ]:
# Import required libraries

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import balanced_accuracy_score

## Import Libraries

Importing all the required libraries at the beginning in advance.

## Load Preprocessed Data

Loading preprocessed data

In [ ]:
# Reading the preprocessed dataset

processed_train = pd.read_parquet(
    "/kaggle/input/datasets/shivamgravity/pgs-s6e6-processed-data-v1/train_processed.parquet"
)
processed_test = pd.read_parquet(
    "/kaggle/input/datasets/shivamgravity/pgs-s6e6-processed-data-v1/test_processed.parquet"
)

## Specifying Configs

Explictly writing settings and parameters for further use.

In [ ]:
# Configs

# Target feature
TARGET = "class"
ID = "id"

# Test ids - used to create submission files after prediction
TEST_ID = processed_test[ID]

# Categorical columns
cat_cols = [
    "spectral_type",
    "galaxy_population"
]
CAT_COL_1 = "spectral_type"
CAT_COL_2 = "galaxy_population"

# CV configs
N_SPLITS = 5
random_state = 42


## Data Preparation For Training & Testing

Splitting target feature from train dataset early, to manage the training further.

Removing the ID feature from train and test dataset both.

In [ ]:
# Preparing the datasets for training and testing purpose

# Removing the id and target feature
X = processed_train.drop([ID,TARGET], axis=1).copy()
y = processed_train[TARGET]

# Removing the id feature
X_test = processed_test.drop([ID], axis=1).copy()

# Number of classes
N_CLASSES = len(np.unique(y))

# Number of columns
NUM_COLS = X.select_dtypes(include=[np.number]).columns.tolist()

## Encoding Categorical Feature

Encoding **TARGET** and **other categorical features**.

It allows LightGBM to access the values in numeric format.

In [ ]:
# Encoding

# Encoding target feature
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y)

# Encoding categorical features beside target feature
feature_encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

    feature_encoders[col] = le

print(feature_encoders)

## Build the model

Building a function to reuse the model architecture.

In [ ]:
# Function to build the model

def build_model():

    num_input = tf.keras.Input(
        shape=(len(NUM_COLS),),
        name="num_input"
    )

    spec_input = tf.keras.Input(
        shape=(1,),
        name="spectral_type"
    )

    pop_input = tf.keras.Input(
        shape=(1,),
        name="galaxy_population"
    )

    # spectral_type (5 categories)
    spec_emb = tf.keras.layers.Embedding(
        input_dim=5,
        output_dim=3
    )(spec_input)

    spec_emb = tf.keras.layers.Flatten()(spec_emb)

    # galaxy_population (2 categories)
    pop_emb = tf.keras.layers.Embedding(
        input_dim=2,
        output_dim=2
    )(pop_input)

    pop_emb = tf.keras.layers.Flatten()(pop_emb)

    x = tf.keras.layers.Concatenate()(
        [num_input, spec_emb, pop_emb]
    )

    x = tf.keras.layers.Dense(256)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation("relu")(x)
    x = tf.keras.layers.Dropout(0.25)(x)

    x = tf.keras.layers.Dense(128)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation("relu")(x)
    x = tf.keras.layers.Dropout(0.20)(x)

    x = tf.keras.layers.Dense(64)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation("relu")(x)

    output = tf.keras.layers.Dense(
        N_CLASSES,
        activation="softmax"
    )(x)

    model = tf.keras.Model(
        inputs=[
            num_input,
            spec_input,
            pop_input
        ],
        outputs=output
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=1e-3
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

## Cross Validation Code

To improve CV Score, I am using **StratifiedKFold**.

It makes sure that every fold has **equal distribution** of **classes**.

In [ ]:
# Manager of dataset distrubtion across folds in CV (Cross Validation)

SKF = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=random_state
)

In [ ]:
# CV code

oof_probs = np.zeros(
    (len(X), N_CLASSES),
    dtype=np.float32
)

test_probs = np.zeros(
    (len(X_test), N_CLASSES),
    dtype=np.float32
)

scores = []

# ============================================================
# CV LOOP
# ============================================================

for fold, (train_idx, valid_idx) in enumerate(
    SKF.split(X, y),
    start=1
):

    print(f"\n{'='*20}")
    print(f"FOLD {fold}")
    print(f"{'='*20}")

    X_train = X.iloc[train_idx].copy()
    X_valid = X.iloc[valid_idx].copy()

    y_train = y[train_idx]
    y_valid = y[valid_idx]

    # --------------------------------------------------------
    # NUMERICAL FEATURES
    # --------------------------------------------------------

    scaler = StandardScaler()

    X_num_train = scaler.fit_transform(
        X_train[NUM_COLS]
    )

    X_num_valid = scaler.transform(
        X_valid[NUM_COLS]
    )

    # --------------------------------------------------------
    # CATEGORICAL FEATURES
    # --------------------------------------------------------

    X_spec_train = (
        X_train[CAT_COL_1]
        .values
        .astype(np.int32)
    )

    X_spec_valid = (
        X_valid[CAT_COL_1]
        .values
        .astype(np.int32)
    )

    X_pop_train = (
        X_train[CAT_COL_2]
        .values
        .astype(np.int32)
    )

    X_pop_valid = (
        X_valid[CAT_COL_2]
        .values
        .astype(np.int32)
    )

    # --------------------------------------------------------
    # MODEL
    # --------------------------------------------------------

    model = build_model()

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True,
        verbose=0
    )

    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=0
    )

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    model.fit(
        [
            X_num_train,
            X_spec_train,
            X_pop_train
        ],
        y_train,
        validation_data=(
            [
                X_num_valid,
                X_spec_valid,
                X_pop_valid
            ],
            y_valid
        ),
        epochs=100,
        batch_size=4096,
        callbacks=[
            early_stop,
            reduce_lr
        ],
        verbose=0
    )

    # --------------------------------------------------------
    # PREDICTION ON THE TEST DATA
    # --------------------------------------------------------

    test_num = scaler.transform(
        X_test[NUM_COLS]
    )
    
    test_spec = (
        X_test[CAT_COL_1]
        .values
        .astype(np.int32)
    )
    
    test_pop = (
        X_test[CAT_COL_2]
        .values
        .astype(np.int32)
    )

    fold_test_probs = model.predict(
        [
            test_num,
            test_spec,
            test_pop
        ],
        batch_size=8192,
        verbose=0
    )

    test_probs += fold_test_probs / SKF.n_splits

    # --------------------------------------------------------
    # VALIDATION PREDICTIONS
    # --------------------------------------------------------

    valid_probs = model.predict(
        [
            X_num_valid,
            X_spec_valid,
            X_pop_valid
        ],
        batch_size=8192,
        verbose=0
    )

    valid_preds = np.argmax(
        valid_probs,
        axis=1
    )

    score = balanced_accuracy_score(
        y_valid,
        valid_preds
    )

    oof_probs[valid_idx] = valid_probs

    scores.append(score)

    print(
        f"Fold {fold}: "
        f"{score:.5f}"
    )

test_probs.shape

### Cross Validation Results

It is the **average** of all the scores we have got in all the folds.

In [ ]:
# Final Scores

oof_preds = np.argmax(
    oof_probs,
    axis=1
)

oof_score = balanced_accuracy_score(
    y,
    oof_preds
)

print("\nCV Scores:", scores)
print(
    f"Mean Balanced Accuracy: "
    f"{np.mean(scores):.5f}"
)
print(
    f"Std Balanced Accuracy : "
    f"{np.std(scores):.5f}"
)
print(
    f"OOF Balanced Accuracy : "
    f"{oof_score:.5f}"
)

## Back to Original Labels

Convert the numeric predictions to original labels as per dataset.

In [ ]:
# Convert probabilities to classes and test labels

test_preds = np.argmax(
    test_probs,
    axis=1
)

test_labels = target_encoder.inverse_transform(
    test_preds
)

## Competition Submission File

Saving the prediction as csv file to submit in the competition.

In [ ]:
# Saving the results

# Creating result dataframe
submission = pd.DataFrame({
    "id": TEST_ID,
    "class": test_labels
})

# Saving the submission file
submission.to_csv("submission.csv", index=False)

print("Submission file saved.")